# Trustworthiness-Annealer: Collapse verhindern, Rate sparen

Ein AUTOENCODIX-Annealer, der β nicht nach einer festen Kurve, sondern nach einer **Preservation-Nebenbedingung** führt:

$$\max_\beta \; \beta \quad \text{so dass} \quad \min_i \tilde m_i \ge 0, \qquad \tilde m_i = \frac{m_i(X,\mu) - \text{Ziel}_i}{\text{Toleranz}_i}$$

Das ist eine ε-Constraint-Formulierung der Rate-Preservation-Abwägung; mit mehreren Metriken wird die Nebenbedingung zur
Tchebycheff-/Min-max-Bedingung (die schlechteste normierte Metrik entscheidet). β ist der Lagrange-Multiplikator, der
Controller macht den Dual-Schritt (verwandt: GECO, Rezende & Viola 2018; ControlVAE, Shao et al. 2020). Der Annealer ist
allgemein für eine oder mehrere Metriken geschrieben, **hier wird nur Trustworthiness verwendet**.

Fragen dieses Notebooks:

1. **Collapse:** Verhindert der Annealer Posterior-Collapse zuverlässig (über Seeds), und ab welchem Sollwert nicht mehr?
2. **Rate:** Wie viel Rate wird gegenüber β = 0 eingespart, ohne dass der Sollwert verletzt wird?
3. **Preservation unter Druck:** Bleibt Preservation bei wenig Informationskosten hoch und fällt erst bei hohen Kosten
   (Knick) oder fällt sie stetig mit der Rate? Gemessen über mehrere k (Trustworthiness, R_NX).
4. **AUC:** Wie folgen Krebs-AUC und Geschlechts-AUC (nur innerhalb gemischter Krebsarten) auf z?
5. **Raum:** Wie verändert sich der Latentraum von getrennten Clustern zu Overlap (labelfrei, `cluster_overlap.py`)?
6. **k des Controllers:** Wie stark hängen Stopp, Rate und Preservation davon ab, mit welchem k gesteuert wird?

Rate, AUC und Labels steuern nichts; der Controller sieht nur X und μ des festen Validierungs-Subsets.

In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Callable
import gc
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from sklearn.manifold import trustworthiness
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "beta_vae_experiments" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if os.environ.get("PYTHONHASHSEED") != "2":
    raise RuntimeError("TCGA braucht PYTHONHASHSEED=2 vor dem Kernel-Start: Kernel 'VAE Beta Control (TCGA)' wählen.")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import autoencodix as acx
from autoencodix.utils._annealer import AnnealingScheduler
from autoencodix.losses.varix_loss import VarixLoss
from zadu.measures import local_continuity_meta_criteria

from beta_vae_experiments.new_beta_varix_experiments import (
    build_varix_config, prepare_fixed_data_splits, save_split_manifest, save_run_loss_history)
from beta_vae_experiments.data_pipeline import build_data_config, save_epoch_snapshots
from beta_vae_experiments.preservation_annealing import _self_first_neighbours, seek_preservation_target, \
    PreservationAnnealingConfig, PreservationStage
from beta_vae_experiments.metrics.information_theoretic import compute_conditional_kl_matrix
from beta_vae_experiments.metrics.latent_usage import PosteriorCollapseConfig
from beta_vae_experiments.metrics.evaluation.downstream import ClassAUCEvaluator, snapshot_tasks
from beta_vae_experiments.live_dashboard import LiveDashboard

OUTPUT = PROJECT_ROOT / "beta_vae_experiments" / "outputs" / "trustworthiness_annealer"
RUNS = OUTPUT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
COLLAPSE = PosteriorCollapseConfig()  # ε = 0,01 Nats, δ = 1 %
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold", "legend.frameon": False})

## 1. Der Annealer

Zwei Klassen, beide über die öffentlichen AUTOENCODIX-Schnittstellen:

* `MaximinAnnealer(AnnealingScheduler)` erbt den AUTOENCODIX-Annealer und überschreibt nur `get_weight`. Er liefert das
  aktuelle Gewicht w ∈ [0, 1]; `VarixLoss` rechnet wie immer `β_eff = β_config · w`. Mit `native=True` gibt er unverändert
  die AUTOENCODIX-Kurve (z. B. `logistic-mid`) zurück und misst nur mit: so laufen die Vergleichsläufe ohne Regelung
  durch denselben Code.
* `annealed_varix_loss(...)` erzeugt eine `VarixLoss`-Unterklasse, die in der Validierung (ohne Gradient) X und μ des festen
  Subsets einsammelt, am Epochenende misst und dem Annealer die Entscheidung überlässt. Der Loss selbst bleibt der von
  AUTOENCODIX.

**Regel** (pro Messung, alle `every` Epochen nach dem Warm-up): Signal s = min_i m̃_i.
s > 1 → β um den Faktor e^Schritt erhöhen; s < −1 → senken; |s| ≤ 1 → halten. Bei jedem Richtungswechsel wird der
log-Schritt halbiert (minimal 0,1 %). Mit einer einzigen Metrik ist das genau `seek_preservation_target` der bisherigen
Läufe (Test unten). Ein ungültiger Messwert zählt als s = −∞ (β runter), nie als Erfolg.

Das Log (`controller_log.csv`) enthält pro Epoche den Zustand, *mit dem die Epoche trainiert wurde*; die Entscheidung aus
einer Messung wirkt ab der nächsten Epoche. Zusätzlich werden pro Epoche Rate und ε-δ-Collapse auf allen
Validierungspatienten mitgeschrieben (nur Diagnose).

In [2]:
@dataclass(frozen=True)
class Constraint:
    """Nebenbedingung m(X, μ) ≥ target für eine höher-ist-besser-Metrik."""
    name: str
    measure: Callable[[np.ndarray, np.ndarray], float]
    target: float
    tolerance: float = 0.01


def trustworthiness_constraint(target, k=20, tolerance=0.01):
    return Constraint(f"trustworthiness_k{k}", lambda x, mu: float(trustworthiness(x, mu, n_neighbors=k)), target, tolerance)


class MaximinAnnealer(AnnealingScheduler):
    """β so groß wie möglich, solange die schlechteste normierte Metrik im Sollband oder darüber liegt."""

    def __init__(self, constraints=(), *, warmup_epochs=100, every=5, growth=1.25, w_min=1e-4,
                 min_change=1e-3, native=False):
        super().__init__()
        self.constraints = tuple(constraints)
        if not self.constraints:
            raise ValueError("Mindestens eine Nebenbedingung (auch native Läufe messen mit).")
        self.warmup_epochs, self.every, self.w_min, self.native = warmup_epochs, every, w_min, native
        self.min_step = float(np.log1p(min_change))
        self.w, self.step, self.direction = 0.0, float(np.log(growth)), 0
        self.state, self.reason = ("native", "") if native else ("warmup", "")

    def get_weight(self, *, epoch_current, total_epoch, func="logistic-mid"):
        if self.native:
            return super().get_weight(epoch_current=epoch_current, total_epoch=total_epoch, func=func)
        return self.w

    def measures_at(self, epoch, total_epochs):
        """epoch ist 0-basiert wie in AUTOENCODIX; gemessen wird nach dem Training dieser Epoche."""
        done = epoch + 1
        return done == self.warmup_epochs or done == total_epochs or (
            done > self.warmup_epochs and (done - self.warmup_epochs) % self.every == 0)

    def evaluate(self, x, mu):
        values, margins = {}, {}
        for constraint in self.constraints:
            try:
                value = constraint.measure(x, mu) if len(np.unique(mu, axis=0)) > 1 else np.nan
            except (ValueError, RuntimeError, ZeroDivisionError):
                value = np.nan
            values[constraint.name] = value
            margins[constraint.name] = (value - constraint.target) / constraint.tolerance if np.isfinite(value) else -np.inf
        return values, margins

    def update(self, margins):
        """Ein Dual-Schritt auf log β. Gibt das Signal s = min_i m̃_i zurück."""
        signal = min(margins.values())
        if self.native:
            return signal
        if abs(signal) <= 1 + 1e-12:
            self.state, self.reason = "hold", "target_band"
            return signal
        direction = 1 if signal > 0 else -1
        if self.direction and direction != self.direction:
            self.step = max(self.step / 2, self.min_step)
        weight = self.w_min if self.w == 0 and direction > 0 else self.w * np.exp(direction * self.step)
        weight = float(np.clip(weight, 0.0, 1.0))
        if direction < 0 and weight < self.w_min:
            weight = 0.0
        bounded = weight == self.w
        self.reason = ("weight_cap" if direction > 0 else "zero_weight_bound") if bounded else (
            "above_target" if direction > 0 else "below_target")
        self.state = "hold" if bounded else "seeking"
        self.w, self.direction = weight, direction
        return signal


def annealed_varix_loss(annealer, folder, valid_ids, positions, on_measure=None):
    """VarixLoss-Unterklasse, die für den Annealer misst. Zustand liegt im Closure, weil AUTOENCODIX den Loss
    für Training und Vorhersage getrennt instanziiert (`loss_type(config=...)`)."""
    folder = Path(folder)
    n_valid = len(valid_ids)
    shared = dict(reference=None, rows=[], logged=set())

    class AnnealedVarixLoss(VarixLoss):
        def __init__(self, config, annealing_scheduler=None):
            if annealing_scheduler is not None:
                raise ValueError("Der Annealer ist bereits gebunden.")
            if config.anneal_function == "no-annealing" or config.default_vae_loss != "kl" or config.loss_reduction != "mean":
                raise ValueError("Braucht KL, mean-Reduktion und eine aktive anneal_function (sonst wird get_weight ignoriert).")
            super().__init__(config, annealing_scheduler=annealer)
            self.epoch, self.offset = None, 0
            self.inputs, self.means, self.logvars = [], [], []

        def flush(self):
            if self.epoch is None or self.epoch in shared["logged"]:
                return
            if self.offset != n_valid:
                raise ValueError("Validierungs-Loader hat nicht die erwartete geordnete Kohorte geliefert.")
            mu, logvar = np.concatenate(self.means), np.concatenate(self.logvars)
            if shared["reference"] is None:
                shared["reference"] = np.concatenate(self.inputs)
                np.savez_compressed(folder / "controller_validation_reference.npz", x=shared["reference"],
                                    sample_ids=np.asarray(valid_ids, dtype=str)[positions], validation_positions=positions)
            total = self.config.epochs
            weight = annealer.get_weight(epoch_current=self.epoch, total_epoch=total, func=self.config.anneal_function)
            kl = compute_conditional_kl_matrix(mu, logvar)
            row = dict(epoch=self.epoch + 1, w=weight, effective_beta=self.config.beta * weight,
                       beta_over_d=self.config.beta * weight / self.config.latent_dim,
                       state=annealer.state, reason=annealer.reason,
                       rate=float(kl.sum(1).mean()),
                       collapsed_dimensions=int(np.sum(np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta)),
                       zero_patients=float((mu == 0).all(1).mean()),
                       metric=",".join(c.name for c in annealer.constraints), metric_value=np.nan, signal=np.nan,
                       target=annealer.constraints[0].target, tolerance=annealer.constraints[0].tolerance,
                       seed=self.config.global_seed, latent_dim=self.config.latent_dim)
            measured = annealer.measures_at(self.epoch, total)
            if measured:
                subset = mu[positions].astype(np.float64)
                values, margins = annealer.evaluate(shared["reference"], subset)
                binding = min(margins, key=margins.get)
                constraint = next(c for c in annealer.constraints if c.name == binding)
                row.update({f"value_{name}": value for name, value in values.items()},
                           metric_value=values[binding], target=constraint.target, tolerance=constraint.tolerance,
                           signal=margins[binding], distinct_mu_share=len(np.unique(subset, axis=0)) / len(subset))
            shared["rows"].append(row)
            shared["logged"].add(self.epoch)
            log = pd.DataFrame(shared["rows"])
            log.to_csv(folder / "controller_log.csv", index=False)
            # Entscheidung erst nach dem Warm-up und nicht nach der letzten trainierten Epoche.
            if measured and self.epoch + 1 >= annealer.warmup_epochs and self.epoch < total - 1:
                annealer.update(margins)
                if on_measure is not None:
                    on_measure(log)
            self.offset, self.inputs, self.means, self.logvars = 0, [], [], []

        def forward(self, model_output, targets, epoch=None, **kwargs):
            if epoch is not None and torch.is_grad_enabled() and epoch != self.epoch:
                self.flush()
                self.epoch = int(epoch)
            if epoch is not None and not torch.is_grad_enabled():
                local = positions[(positions >= self.offset) & (positions < self.offset + len(targets))] - self.offset
                if shared["reference"] is None and len(local):
                    self.inputs.append(targets[local].detach().cpu().numpy())
                self.means.append(model_output.latent_mean.detach().cpu().numpy())
                self.logvars.append(model_output.latent_logvar.detach().cpu().numpy())
                self.offset += len(targets)
            result = super().forward(model_output=model_output, targets=targets, epoch=epoch, **kwargs)
            if epoch == self.config.epochs - 1 and not torch.is_grad_enabled() and self.offset == n_valid:
                self.flush()
            return result

    return AnnealedVarixLoss

**Test:** Mit einer Nebenbedingung trifft `MaximinAnnealer.update` dieselben Entscheidungen wie der bisherige Controller
`seek_preservation_target` (gleiche Messfolge, gleiche Gewichte). Die Scores sind künstlich und decken Erhöhen, Halten,
Senken und Richtungswechsel ab.

In [3]:
rng = np.random.default_rng(0)
scores = np.concatenate([np.linspace(.97, .9, 30), .9 + .03 * rng.standard_normal(60), np.linspace(.88, .8, 10)])
new = MaximinAnnealer([Constraint("t", None, .9, .01)])
old_policy = PreservationAnnealingConfig(target=.9, rule="target", tolerance=.01, w_min=1e-4, growth=1.25)
old = PreservationStage(state="seeking")
for score in scores:
    new.update({"t": (score - .9) / .01})
    old = seek_preservation_target(old, score, .9, old_policy)
    assert np.isclose(new.w, old.w, rtol=1e-12, atol=0) and new.state == old.state, (score, new.w, old.w)
print(f"{len(scores)} Entscheidungen identisch; letztes w = {new.w:.3g}")

100 Entscheidungen identisch; letztes w = 0.0238


## 2. Daten und Versuchsplan

Gleiche Vorverarbeitung wie alle bisherigen TCGA-Läufe (AUTOENCODIX, MINMAX-Skalierung, k_filter = 2000, Split-Seed 2),
d = 8, 600 Epochen, Lernrate 5·10⁻⁴, Controller wie im Pilot: Warm-up 100 Epochen (β = 0), Messung alle 5 Epochen auf
800 festen Validierungspatienten, Startgewicht 10⁻⁴, Wachstum 1,25, β/d höchstens 0,015. Snapshots alle 20 Epochen.

| Block | Läufe | Wofür |
|---|---|---|
| A | Trustworthiness (k = 20), Ziel 0,95 / 0,925 / 0,90 / 0,875 / 0,85 / 0,80 / 0,75 / 0,70 | Fragen 1–5: Front Rate ↔ Preservation ↔ AUC |
| B | Ziel 0,90 mit Controller-k = 10 / 40 / 200 | Frage 6 (k = 20 steckt in A) |
| C | ohne Regelung: AUTOENCODIX `logistic-mid` bis β/d = 1,5·10⁻³ bzw. 4·10⁻³ | Gegenprobe: gleicher β-Bereich ohne Rückkopplung |

Jeder Block mit den Seeds 17, 23 und 31; zuerst alle Läufe mit Seed 17, damit früh ein vollständiges Bild entsteht.
Fertige Läufe (`run.json` im Lauf-Ordner) werden übersprungen; die Auswertung unten nimmt, was fertig ist.

In [4]:
EPOCHS, LATENT_DIM, SNAPSHOT_EVERY = 600, 8, 20
BETA_OVER_D_MAX = 0.015
CONTROLLER = dict(warmup_epochs=100, every=5, growth=1.25, w_min=1e-4)
N_SUBSET, SUBSET_SEED = 800, 31
SEEDS = (17, 23, 31)
TARGETS = (0.95, 0.925, 0.90, 0.875, 0.85, 0.80, 0.75, 0.70)
CONTROLLER_KS = (10, 40, 200)
FIXED_BETA_OVER_D = (1.5e-3, 4e-3)


@dataclass(frozen=True)
class RunSpec:
    block: str
    seed: int
    target: float = np.nan
    k: int = 20
    fixed_beta_over_d: float = np.nan

    @property
    def name(self):
        if self.block == "C":
            return f"C_fixed_bd{self.fixed_beta_over_d:.1e}_s{self.seed}"
        return f"{self.block}_T{self.target:.3f}_k{self.k}_s{self.seed}"


grid = []
for seed in SEEDS:
    grid += [RunSpec("A", seed, target) for target in TARGETS]
    grid += [RunSpec("B", seed, 0.90, k) for k in CONTROLLER_KS]
    grid += [RunSpec("C", seed, fixed_beta_over_d=value) for value in FIXED_BETA_OVER_D]
plan = pd.DataFrame([dict(name=s.name, block=s.block, seed=s.seed, target=s.target, k=s.k,
                          fixed_beta_over_d=s.fixed_beta_over_d, fertig=(RUNS / s.name / "run.json").exists()) for s in grid])
print(f"{len(plan)} Läufe, davon fertig: {plan.fertig.sum()}")

dataset = dict(id="tcga", split_seed=2, options=dict(
    data_root="data/raw/TCGAPanCancerAutoencodix", rna_file="data_mrna_seq_v2_rsem_formatted.parquet",
    methylation_file="data_methylation_per_gene_formatted.parquet", clinical_file="data_clinical_formatted.parquet"))
dataset["options"]["data_root"] = str((PROJECT_ROOT / dataset["options"]["data_root"]).resolve())
TRAINING = dict(epochs=EPOCHS, anneal_function="logistic-mid", device="cuda", checkpoint_interval=SNAPSHOT_EVERY,
                snapshots=dict(enabled=True, every=SNAPSHOT_EVERY), downstream_targets=["CANCER_TYPE_ACRONYM", "SEX"])
RUN_DEFAULTS = dict(latent_dim=LATENT_DIM, loss_reduction="mean", learning_rate=5e-4, loss="kl")

data_config = build_data_config(PROJECT_ROOT, "tcga", dataset["options"])
splits = prepare_fixed_data_splits(None, build_varix_config(
    data_config, "tcga", TRAINING, dict(RUN_DEFAULTS, beta=0.0), global_seed=dataset["split_seed"]))
save_split_manifest(splits, OUTPUT)
VALID_IDS = np.asarray(splits.valid.sample_ids, dtype=str)
POSITIONS = np.sort(np.random.default_rng(SUBSET_SEED).choice(len(VALID_IDS), N_SUBSET, replace=False))
print(f"Train {len(splits.train.sample_ids)}, Validierung {len(VALID_IDS)}, Controller-Subset {len(POSITIONS)}")

39 Läufe, davon fertig: 0
reading parquet: /home/nickok/Projects/MasterThesis/autoencodix_package/data/raw/TCGAPanCancerAutoencodix/data_mrna_seq_v2_rsem_formatted.parquet
reading parquet: /home/nickok/Projects/MasterThesis/autoencodix_package/data/raw/TCGAPanCancerAutoencodix/data_methylation_per_gene_formatted.parquet
reading parquet: /home/nickok/Projects/MasterThesis/autoencodix_package/data/raw/TCGAPanCancerAutoencodix/data_clinical_formatted.parquet
anno key: paired
Train 7009, Validierung 1001, Controller-Subset 800


## 3. Training

Läuft im Notebook-Kernel mit dem nativen AUTOENCODIX-Trainer (`acx.Varix(..., loss_type=...)`). Die Live-Anzeige zeigt für den
laufenden Lauf β/d, das Steuersignal mit Sollband und die Rate mit ε-δ-Collapse; sie wird nach jeder Messung neu gezeichnet.
Nach jedem Lauf werden die Snapshots gespeichert und das Live-Dashboard (`live_dashboard.py`) rechnet für jeden Snapshot den
labelfreien Cluster-Overlap (`live_metrics.csv`, `live_summary.md` im Lauf-Ordner).
Ein Lauf mit 600 Epochen dauert etwa 16 Minuten, dazu etwa 2 Minuten Auswertung.

In [ ]:
RUN_TRAINING = True

live_handle = display(Markdown("_Noch keine Messung._"), display_id=True)


def live_plot(spec):
    def draw(log):
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
        axes[0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", color="black")
        axes[0].set(yscale="log", title="β/d", xlabel="Epoche")
        measured = log.dropna(subset=["metric_value"])
        axes[1].plot(measured.epoch, measured.metric_value, "o-", ms=2.5)
        if not spec.block == "C":
            axes[1].axhspan(spec.target - .01, spec.target + .01, alpha=.15)
        axes[1].set(title=f"Trustworthiness (k = {spec.k})", xlabel="Epoche")
        axes[2].plot(log.epoch, log.rate, lw=1.2)
        collapsed = log[log.collapsed_dimensions > 0]
        axes[2].plot(collapsed.epoch, collapsed.rate, "x", color="black", ms=4, label="≥ 1 kollabierte Dim.")
        axes[2].set(yscale="log", title="Rate (alle Validierungspatienten)", xlabel="Epoche")
        if len(collapsed):
            axes[2].legend(fontsize=8)
        fig.suptitle(spec.name)
        fig.tight_layout()
        live_handle.update(fig)
        plt.close(fig)
    return draw


def train(spec):
    folder = RUNS / spec.name
    folder.mkdir(parents=True, exist_ok=True)
    started = time.time()
    if spec.block == "C":
        beta_max = spec.fixed_beta_over_d
        annealer = MaximinAnnealer([trustworthiness_constraint(0.9, spec.k)], native=True, **CONTROLLER)
    else:
        beta_max = BETA_OVER_D_MAX
        annealer = MaximinAnnealer([trustworthiness_constraint(spec.target, spec.k)], **CONTROLLER)
    config = build_varix_config(data_config, "tcga", TRAINING, dict(RUN_DEFAULTS, beta=beta_max * LATENT_DIM),
                                global_seed=spec.seed)
    (folder / "controller_config.json").write_text(json.dumps(dict(
        spec=spec.__dict__, beta_over_d_max=beta_max, controller=CONTROLLER, n_subset=N_SUBSET, subset_seed=SUBSET_SEED,
        epochs=EPOCHS, anneal_function=config.anneal_function, native=annealer.native), indent=2, default=float) + "\n")
    loss_type = annealed_varix_loss(annealer, folder, VALID_IDS, POSITIONS, on_measure=live_plot(spec))
    varix = acx.Varix(data=splits, config=config, loss_type=loss_type, evaluator=ClassAUCEvaluator)
    result = varix.run()
    save_run_loss_history(result, folder)
    save_epoch_snapshots(varix, folder, every=SNAPSHOT_EVERY, downstream=False)
    trained = time.time() - started
    del varix, result
    gc.collect()
    torch.cuda.empty_cache()
    LiveDashboard(folder).update_metrics()
    (folder / "run.json").write_text(json.dumps(dict(spec.__dict__, name=spec.name, train_seconds=trained,
                                                     total_seconds=time.time() - started), default=float) + "\n")


if RUN_TRAINING:
    todo = [spec for spec in grid if not (RUNS / spec.name / "run.json").exists()]
    for seed in SEEDS:  # erst alle Läufe eines Seeds
        for spec in [s for s in todo if s.seed == seed]:
            print(f"{time.strftime('%H:%M')} Lauf {spec.name}", flush=True)
            train(spec)
print("fertig:", sum((RUNS / s.name / "run.json").exists() for s in grid), "von", len(grid))

## 4. Auswertung je Lauf

Pro Lauf und Snapshot (alle 40 Epochen, dazu Ende Warm-up und Laufende) auf den **gespeicherten Snapshots**:

* Rate (mittlere KL, Nats), kollabierte Dimensionen nach ε-δ, Patienten mit μ = 0 in allen Dimensionen
* Trustworthiness (k = 20) auf dem Controller-Subset, X → μ
* AUC auf z (logistische Regression, Train → Validation): Krebsart (33) und Geschlecht **nur innerhalb gemischter
  Krebsarten** (`snapshot_tasks`, `sex_mixed_cancers`), weil die Krebsart Geschlecht sonst schon zu AUC 0,83 vorhersagt
* Labelfreier Cluster-Overlap aus `live_metrics.csv`: Anteil der 20 nächsten Nachbarn im eigenen X-Cluster (μ und z), ARI

Am Laufende zusätzlich der k-Sweep: Trustworthiness und R_NX(k) = (n−1)·LCMC/(n−1−k) für k = 5 … 200 auf μ und z.
Ergebnisse werden pro Lauf in `analysis_trajectory.csv` und `analysis_final.json` gespeichert und nur einmal gerechnet.

In [ ]:
K_SWEEP = (5, 10, 20, 40, 80, 200)


def load_snapshot(folder, epoch, split):
    with np.load(folder / "epoch_snapshots" / f"epoch_{epoch:05d}_{split}.npz") as saved:
        return {key: saved[key] for key in saved.files} | dict(sample_ids=saved["sample_ids"].astype(str))


def labels_of(folder, split):
    labels = pd.read_parquet(folder / "epoch_snapshots" / f"labels_{split}.parquet")
    labels.index = labels.index.astype(str)
    return labels


def auc_on_z(folder, epoch):
    """Krebs-AUC und Geschlechts-AUC innerhalb gemischter Krebsarten auf z (Train → Validation)."""
    snaps = {split: load_snapshot(folder, epoch, split) for split in ("train", "valid")}
    labels = {split: labels_of(folder, split).loc[snaps[split]["sample_ids"]] for split in snaps}
    tasks = {task["name"]: task for task in snapshot_tasks(labels)}
    scores = {}
    for name, task in (("auc_cancer", tasks["cancer_type"]), ("auc_sex_mixed", tasks["sex_mixed_cancers"])):
        frames = {s: pd.DataFrame(snaps[s]["z"], index=snaps[s]["sample_ids"])[task["keep"][s]] for s in snaps}
        partition = pd.concat([pd.DataFrame(dict(SAMPLE_ID=f.index, SPLIT=s)) for s, f in frames.items()], ignore_index=True)
        annotation = pd.concat([task["y"][s][task["keep"][s]].rename(task["target"]) for s in snaps]).to_frame()
        result = ClassAUCEvaluator()._single_ml_presplit(partition, pd.concat(frames.values()), annotation, task["target"],
                                                         LogisticRegression(max_iter=1000), "roc_auc_ovo",
                                                         "classification", top_k_classes=None)
        scores[name] = float(result.loc[result.score_split.eq("valid"), "value"].iloc[0])
    return scores


def r_nx(x, embedding, k):
    x_neighbours, _ = _self_first_neighbours(x, k)
    neighbours, _ = _self_first_neighbours(embedding, k)
    lcmc = local_continuity_meta_criteria.measure(x, embedding, k=k, knn_info=(x_neighbours, neighbours))["lcmc"]
    return (len(x) - 1) * lcmc / (len(x) - 1 - k)


def analyse_run(spec):
    folder = RUNS / spec.name
    trajectory_path, final_path = folder / "analysis_trajectory.csv", folder / "analysis_final.json"
    if trajectory_path.exists() and final_path.exists():
        return pd.read_csv(trajectory_path), json.loads(final_path.read_text())
    controller = pd.read_csv(folder / "controller_log.csv").set_index("epoch")
    live = pd.read_csv(folder / "live_metrics.csv").set_index("epoch")
    with np.load(folder / "controller_validation_reference.npz") as saved:
        x, reference_ids = saved["x"].astype(np.float64), saved["sample_ids"].astype(str)
    epochs = sorted({CONTROLLER["warmup_epochs"], EPOCHS} | set(range(40, EPOCHS + 1, 40)))
    rows = []
    for epoch in epochs:
        valid = load_snapshot(folder, epoch, "valid")
        positions = pd.Index(valid["sample_ids"]).get_indexer(reference_ids)
        mu = valid["mu"][positions].astype(np.float64)
        kl = compute_conditional_kl_matrix(valid["mu"], valid["logvar"])
        rows.append(dict(epoch=epoch, beta_over_d=controller.loc[epoch, "beta_over_d"], rate=float(kl.sum(1).mean()),
                         collapsed_dimensions=int(np.sum(np.mean(kl < COLLAPSE.primary_epsilon, axis=0) >= 1 - COLLAPSE.delta)),
                         zero_patients=float((valid["mu"] == 0).all(1).mean()),
                         distinct_mu_share=len(np.unique(mu, axis=0)) / len(mu),
                         trustworthiness=float(trustworthiness(x, mu, n_neighbors=20)) if len(np.unique(mu, axis=0)) > 1 else np.nan,
                         **auc_on_z(folder, epoch),
                         same_cluster_mu=live.loc[epoch, "same_cluster_neighbours_latent"],
                         same_cluster_z=live.loc[epoch, "same_cluster_neighbours_z"],
                         same_cluster_x=live.loc[epoch, "same_cluster_neighbours_x"], chance=live.loc[epoch, "chance"],
                         recovery_ari=live.loc[epoch, "recovery_ari"], n_clusters_latent=live.loc[epoch, "n_clusters_latent"]))
    trajectory = pd.DataFrame(rows)
    valid = load_snapshot(folder, EPOCHS, "valid")
    positions = pd.Index(valid["sample_ids"]).get_indexer(reference_ids)
    final = dict(name=spec.name)
    for space in ("mu", "z"):
        values = valid[space][positions].astype(np.float64)
        for k in K_SWEEP:
            degenerate = len(np.unique(values, axis=0)) <= k
            final[f"t_{space}_k{k}"] = np.nan if degenerate else float(trustworthiness(x, values, n_neighbors=k))
            final[f"rnx_{space}_k{k}"] = np.nan if degenerate else float(r_nx(x, values, k))
    trajectory.to_csv(trajectory_path, index=False)
    final_path.write_text(json.dumps(final, indent=1) + "\n")
    return trajectory, final


done = [spec for spec in grid if (RUNS / spec.name / "run.json").exists()]
trajectories, finals = [], []
for spec in done:
    trajectory, final = analyse_run(spec)
    trajectories.append(trajectory.assign(**{k: v for k, v in spec.__dict__.items()}, name=spec.name))
    finals.append(final)
trajectories = pd.concat(trajectories, ignore_index=True)
warmup = trajectories[trajectories.epoch.eq(CONTROLLER["warmup_epochs"])].set_index("name")
last = trajectories[trajectories.epoch.eq(EPOCHS)].set_index("name")
ever_collapsed = trajectories[trajectories.epoch > CONTROLLER["warmup_epochs"]].groupby("name").collapsed_dimensions.max()
results = last.join(pd.DataFrame(finals).set_index("name")).assign(
    rate_warmup=warmup.rate, t_warmup=warmup.trustworthiness, auc_cancer_warmup=warmup.auc_cancer,
    auc_sex_mixed_warmup=warmup.auc_sex_mixed, max_collapsed_dimensions=ever_collapsed)
results["rate_saving"] = 1 - results.rate / results.rate_warmup
results.to_csv(OUTPUT / "results.csv")
print(f"{len(results)} Läufe ausgewertet")

## 5. Frage 1 und 2: Collapse und eingesparte Rate

Pro Sollwert über die Seeds: End-β/d, Rate am Ende des Warm-ups (β = 0) und am Laufende, eingesparte Rate
(1 − Rate_Ende / Rate_Warm-up), Anzahl Seeds mit mindestens einer kollabierten Dimension (ε-δ, irgendwann nach dem
Warm-up), Patienten bei μ = 0 am Ende und Trustworthiness am Ende. Block C ohne Regelung zum Vergleich.

In [ ]:
def setting(row):
    if row.block == "C":
        return f"fest β/d = {row.fixed_beta_over_d:.1e}"
    return f"Ziel {row.target:.3f}, k = {int(row.k)}"


results["setting"] = results.apply(setting, axis=1)
order = results.sort_values(["block", "target", "k", "fixed_beta_over_d"], ascending=[True, False, True, True]).setting.unique()
summary = results.groupby("setting").agg(
    seeds=("seed", "size"), beta_over_d=("beta_over_d", "median"), rate_warmup=("rate_warmup", "median"),
    rate=("rate", "median"), rate_min=("rate", "min"), rate_max=("rate", "max"),
    rate_saving=("rate_saving", "median"), seeds_with_collapse=("max_collapsed_dimensions", lambda s: int((s > 0).sum())),
    collapsed_end=("collapsed_dimensions", "max"), zero_patients=("zero_patients", "max"),
    trustworthiness=("trustworthiness", "median"), t_warmup=("t_warmup", "median")).reindex(order)
display(summary.style.format({"beta_over_d": "{:.2e}", "rate_warmup": "{:.2f}", "rate": "{:.3f}", "rate_min": "{:.3f}",
                              "rate_max": "{:.3f}", "rate_saving": "{:.1%}", "zero_patients": "{:.1%}",
                              "trustworthiness": "{:.3f}", "t_warmup": "{:.3f}"})
        .set_caption("Median über Seeds (Rate-Spanne min–max); Collapse = Seeds mit ≥ 1 kollabierter Dimension nach dem Warm-up"))
summary.to_csv(OUTPUT / "summary_by_setting.csv")

## 6. Frage 3 und 4: Preservation und AUC gegen die Rate

Jeder Punkt ist ein Lauf am Ende (600 Epochen); die Linien sind die Verläufe des Seed-17-Laufs je Sollwert über die
Snapshots (von rechts, hohe Rate am Warm-up-Ende, nach links). Graue Kreuze: Block C ohne Regelung. Die x-Achse ist
logarithmisch; die senkrechte Linie bei 0,5 Nats markiert SNR = 1 (E‖μ‖² = σ²), darunter beginnt in den bisherigen
Läufen der massive Collapse.

In [ ]:
A = results[results.block.eq("A")]
C = results[results.block.eq("C")]
colors = dict(zip(TARGETS, plt.cm.viridis(np.linspace(0, .9, len(TARGETS)))))
panels = [("trustworthiness", "Trustworthiness (k = 20, μ)"), ("rnx_z_k10", "R_NX k = 10 (z)"),
          ("rnx_z_k200", "R_NX k = 200 (z)"), ("auc_cancer", "Krebs-AUC (z)"),
          ("auc_sex_mixed", "Geschlechts-AUC gemischte Krebsarten (z)"), ("same_cluster_z", "Nachbarn im eigenen X-Cluster (z)")]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for axis, (column, title) in zip(axes.ravel(), panels):
    for target in TARGETS:
        group = A[A.target.eq(target)]
        axis.scatter(group.rate, group[column], color=colors[target], s=40, label=f"Ziel {target:g}", zorder=3)
        path = trajectories[(trajectories.block.eq("A")) & trajectories.target.eq(target) & trajectories.seed.eq(17)
                            & (trajectories.epoch >= CONTROLLER["warmup_epochs"])]
        if column in path:
            axis.plot(path.rate, path[column], "-", color=colors[target], lw=.8, alpha=.6)
    axis.scatter(C.rate, C[column], marker="x", color="gray", s=50, label="ohne Regelung (C)", zorder=3)
    axis.axvline(.5, color="red", ls=":", lw=.8)
    axis.set(xscale="log", xlabel="Rate am Laufende [Nats]", title=title)
axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT / "front_rate_preservation_auc.png", dpi=150)
plt.show()

### 6.1 Preservation über k

Wie stark fällt die Erhaltung pro Nachbarschaftsgröße? R_NX und Trustworthiness am Laufende gegen den Sollwert, je k eine
Linie (Median über Seeds), auf μ und z. Links (hoher Sollwert) ist wenig Druck, rechts viel.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True)
for axis, (prefix, title) in zip(axes, [("t_mu", "Trustworthiness μ"), ("t_z", "Trustworthiness z"),
                                        ("rnx_mu", "R_NX μ"), ("rnx_z", "R_NX z")]):
    medians = A.groupby("target")[[f"{prefix}_k{k}" for k in K_SWEEP]].median()
    for k, color in zip(K_SWEEP, plt.cm.plasma(np.linspace(0, .85, len(K_SWEEP)))):
        axis.plot(medians.index, medians[f"{prefix}_k{k}"], "o-", ms=3, color=color, label=f"k = {k}")
    axis.set(title=title, xlabel="Trustworthiness-Sollwert (k = 20)")
    axis.invert_xaxis()
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / "preservation_over_k.png", dpi=150)
plt.show()
k_table = A.groupby("target")[[f"rnx_z_k{k}" for k in K_SWEEP] + ["rate"]].median().sort_index(ascending=False)
display(k_table.style.format("{:.3f}").set_caption("R_NX auf z am Laufende (Median über Seeds) und Rate"))

## 7. Frage 5: Vom Cluster zum Overlap

Labelfrei: k-means auf X (Controller-Subset) definiert die Referenz-Cluster. Gezeigt ist pro Sollwert (Seed 17) die UMAP
von z am Laufende (AUTOENCODIX-Stil, eingefärbt nach Krebsart) und darunter der automatisch erzeugte Overlap-Text.

In [ ]:
show_targets = [t for t in (0.95, 0.90, 0.80, 0.70) if f"A_T{t:.3f}_k20_s17" in results.index]
fig, axes = plt.subplots(1, len(show_targets), figsize=(6 * len(show_targets), 4.6), squeeze=False)
for axis, target in zip(axes[0], show_targets):
    name = f"A_T{target:.3f}_k20_s17"
    axis.imshow(LiveDashboard(RUNS / name).umap_image(EPOCHS))
    axis.set_axis_off()
    row = results.loc[name]
    axis.set_title(f"Ziel {target:g}: Rate {row.rate:.2f} Nats, eigener Cluster {row.same_cluster_z:.0%}", fontsize=10)
fig.tight_layout()
fig.savefig(OUTPUT / "umap_by_target.png", dpi=150)
plt.show()
for target in show_targets:
    name = f"A_T{target:.3f}_k20_s17"
    texts = json.loads((RUNS / name / "live_descriptions.json").read_text())
    display(Markdown(f"**Ziel {target:g}** (Epoche {EPOCHS})\n\n" + texts[str(EPOCHS)].replace("\n", "\n\n")))

## 8. Frage 6: k des Controllers

Gleiches Ziel 0,90, gesteuert mit Trustworthiness bei k = 10, 20, 40 oder 200. Verglichen werden End-β/d, Rate und die
Erhaltung bei festen k (unabhängig vom Steuer-k).

In [ ]:
B = results[results.target.eq(0.90) & results.block.isin(["A", "B"])]
columns = ["beta_over_d", "rate", "rate_saving", "max_collapsed_dimensions", "t_mu_k20", "rnx_z_k10", "rnx_z_k40",
           "rnx_z_k200", "auc_cancer", "auc_sex_mixed", "same_cluster_z"]
k_summary = B.groupby("k")[columns].median()
display(k_summary.style.format("{:.3f}").format({"beta_over_d": "{:.2e}", "rate_saving": "{:.1%}"})
        .set_caption("Ziel 0,90: Median über Seeds je Controller-k"))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for k, group in B.groupby("k"):
    path = trajectories[trajectories.name.isin(group.index) & trajectories.seed.eq(17)]
    log = pd.read_csv(RUNS / group.index[group.seed.eq(17).to_numpy()][0] / "controller_log.csv") if (group.seed == 17).any() else None
    if log is not None:
        axes[0].step(log.epoch, log.beta_over_d.clip(lower=1e-9), where="post", label=f"k = {k}")
    axes[1].plot(path.epoch, path.rate, "o-", ms=3, label=f"k = {k}")
    axes[2].plot(path.epoch, path.trustworthiness, "o-", ms=3, label=f"k = {k}")
axes[0].set(yscale="log", title="β/d (Seed 17)", xlabel="Epoche")
axes[1].set(yscale="log", title="Rate", xlabel="Epoche")
axes[2].set(title="Trustworthiness bei k = 20", xlabel="Epoche")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT / "controller_k.png", dpi=150)
plt.show()